# GE-Sentinel — exploratory data analysis
Run `python -m ge_sentinel.cli demo` first so the database is populated.
Real rows are tagged `seed_real`/`live`; synthetic rows are tagged `synth`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))
import pandas as pd
from ge_sentinel import db
from ge_sentinel.ingest import load_prices, gap_report
engine = db.init_db()
prices = load_prices(engine)
prices.groupby('source')['item_id'].agg(['count', 'nunique'])

In [ ]:
# Real seed item: Abyssal whip (4151)
whip = load_prices(engine, [4151]).sort_values('ts')
whip['mid'] = whip[['avg_high','avg_low']].mean(axis=1)
ax = whip.set_index(pd.to_datetime(whip['ts'], unit='s'))['mid'].plot(figsize=(10,3), title='Abyssal whip — real 5m mid price')
ax.set_ylabel('gp')

In [ ]:
# Data quality: genuine missing buckets in the real excerpt
gap_report(engine, 4151)

In [ ]:
# Feature view on one synthetic item with an injected event
from ge_sentinel.features import build_features
import sqlalchemy as sa
truth = pd.read_sql(sa.select(db.truth_events), engine)
ev = truth[truth.kind=='pump_dump'].sort_values('magnitude').iloc[-1]
f = build_features(load_prices(engine, [int(ev.item_id)]))
w = f[f.ts.between(ev.start_ts-6000, ev.end_ts+6000)].set_index(pd.to_datetime(f[f.ts.between(ev.start_ts-6000, ev.end_ts+6000)].ts, unit='s'))
w[['vol_z']].plot(figsize=(10,2.5), title=f'vol_z around injected pump (item {int(ev.item_id)})')

Next: `detectors.ensemble` scores, `evaluate.evaluate` against `truth_events`,
and the alert memos under `data/outputs/memos/`.